In [1]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: cryptography
    Found existing installation: cryptography 50.0.1
    Uninstalling cryptography-50.0.1:
      Successfully uninstalled cryptography-50.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.4.0 requires cryptography<51,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.
bigframes 2.49.0 requires pyarrow>=23.0.1, but you have pyarrow 21.

In [111]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [112]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

**Define Ability**



In [118]:
from eyepop import EyePopSdk
from eyepop.data.data_types import (
    InferRuntimeConfig,
    VlmAbilityGroupCreate,
    VlmAbilityCreate,
    TransformInto,
)
from eyepop.worker.worker_types import InferenceComponent, Pop


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.pedestrian-safety-violations",
        description=(
            "Detects when a pedestrian enters, occupies, touches, "
            "or moves alongside a marked forklift or vehicle safety zone"
        ),
        worker_release="qwen3-instruct",
        text_prompt = """
        Look at all people visible in the image and the yellow boundary lines on the floor.

        Determine whether the feet of ANY person are inside the area enclosed by the yellow lines.

        Output only one label:
        Inside_Safety_Zone
        Outside_Safety_Zone

        Output Inside_Safety_Zone if at least one person's feet are inside the enclosed area.
        Output Outside_Safety_Zone only if every person's feet are outside the enclosed area.
        """
        ,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=5,
            fps=1,
            image_size=640,
        ),
        is_public=False,
    )
]





**Create Ability**

In [120]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06abee73743f71b4800068030f0eebb0 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.find-events.pedestrian-safety-violations', tag='1.0.48'), AbilityAliasEntry(alias='datasciencealliance-org.find-events.pedestrian-safety-violations', tag='latest')]


**Evaluate on a single video**

In [115]:
from pathlib import Path
import json
from collections import Counter, defaultdict
import cv2

from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop


# ---------------------------------------------------------
# Ability setup
# ---------------------------------------------------------

pop = Pop(
    components=[
        InferenceComponent(
            ability=f"{NAMESPACE_PREFIX}.find-events.pedestrian-safety-violations:latest",
        )
    ]
)

video_path = "/content/vid1.mp4"
sample_video_path = Path(video_path)


# ---------------------------------------------------------
# Get video information
# ---------------------------------------------------------

cap = cv2.VideoCapture(str(sample_video_path))

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

cap.release()

video_duration = (
    frame_count / fps
    if fps and frame_count
    else None
)


# ---------------------------------------------------------
# Store inference results
# ---------------------------------------------------------

class_counts = Counter()
timeline = []
raw_results_sample = []


# ---------------------------------------------------------
# Run inference
# ---------------------------------------------------------

with EyePopSdk.workerEndpoint(
    api_key=EYEPOP_API_KEY,
) as endpoint:

    endpoint.set_pop(pop)

    job = endpoint.upload(sample_video_path)

    while result := job.predict():

        # Keep a few raw predictions for inspection.
        if len(raw_results_sample) < 3:
            raw_results_sample.append(result)

        # The ability may return transformed classes.
        for class_item in result.get("classes", []):

            label = (
                class_item
                .get("classLabel", "")
                .strip()
            )

            confidence = class_item.get("confidence")

            if not label:
                continue

            class_counts[label] += 1

            timeline.append({
                "seconds": result.get("seconds"),
                "label": label,
                "confidence": confidence,
            })

        # Fallback in case the ability returns text instead
        # of a transformed class.
        for text_item in result.get("texts", []):

            label = (
                text_item
                .get("text", "")
                .strip()
            )

            if not label:
                continue

            class_counts[label] += 1

            timeline.append({
                "seconds": result.get("seconds"),
                "label": label,
                "confidence": None,
            })


# ---------------------------------------------------------
# Raw output sample
# ---------------------------------------------------------

print("\n=== RAW RESULT SAMPLE ===")

print(
    json.dumps(
        raw_results_sample,
        indent=2,
    )
)


# ---------------------------------------------------------
# Classification by second
# ---------------------------------------------------------

second_states = defaultdict(list)

for item in timeline:
    seconds = item.get("seconds")

    if seconds is None:
        continue

    second = int(seconds)
    second_states[second].append(item["label"])


print("\n=== CLASSIFICATION BY SECOND ===")

for second in sorted(second_states):

    counts = Counter(second_states[second])
    total = sum(counts.values())

    inside = counts["Inside_Safety_Zone"]
    outside = counts["Outside_Safety_Zone"]

    inside_ratio = (
        inside / total
        if total
        else 0
    )

    print(
        f"{second:02d}s -> "
        f"Inside Safety Zone ratio={inside_ratio:.2f} "
        f"(Inside={inside}, Outside={outside})"
    )


# ---------------------------------------------------------
# Overall classification counts
# ---------------------------------------------------------

print("\n=== OVERALL CLASSIFICATION COUNTS ===")

for label, count in class_counts.most_common():
    print(f"{label}: {count}")


# ---------------------------------------------------------
# Video information
# ---------------------------------------------------------

if video_duration is not None:
    print(
        f"\nActual video duration: "
        f"{video_duration:.2f} seconds"
    )


print("\nDone")


=== RAW RESULT SAMPLE ===
[
  {
    "classes": [
      {
        "classLabel": "Outside_Safety_Zone",
        "confidence": 0.9889
      }
    ],
    "compute_units": 0.058,
    "duration": 41666666,
    "seconds": 0.083333333,
    "source_height": 720,
    "source_id": "c67493af-bdea-11f1-aa06-f25500fb30ea",
    "source_width": 1280,
    "system_timestamp": 1790895153251522000,
    "timestamp": 83333333
  },
  {
    "classes": [
      {
        "classLabel": "Outside_Safety_Zone",
        "confidence": 0.9887
      }
    ],
    "compute_units": 0.058,
    "duration": 41666666,
    "seconds": 0.125,
    "source_height": 720,
    "source_id": "c67493af-bdea-11f1-aa06-f25500fb30ea",
    "source_width": 1280,
    "system_timestamp": 1790895153251962000,
    "timestamp": 125000000
  },
  {
    "classes": [
      {
        "classLabel": "Outside_Safety_Zone",
        "confidence": 0.9901
      }
    ],
    "compute_units": 0.058,
    "duration": 41666666,
    "seconds": 0.166666666,
    "s